# Patrón Estructural: Facade

## Introducción
El patrón Facade proporciona una interfaz simplificada a una biblioteca, un framework o cualquier otro grupo complejo de clases.

## Objetivos
- Comprender cómo simplificar el acceso a sistemas complejos.
- Identificar cuándo es útil el patrón Facade.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Estación Meteorológica**
Supón que una app meteorológica necesita consultar datos de sensores, procesar información y mostrar resultados. El patrón Facade permite exponer una interfaz simple para el usuario, ocultando la complejidad interna.

**¿Dónde se usa en proyectos reales?**
En bibliotecas de gráficos, sistemas de bases de datos, APIs de hardware, etc.

## Sin patrón Facade (forma errónea)
El cliente debe interactuar con múltiples clases y coordinar sus acciones manualmente.

In [1]:
class Sensor:
    def leer(self) -> float:
        return 25

class Procesador:
    def procesar(self, dato: float) -> float:
        return dato * 1.8 + 32

class Visualizador:
    def mostrar(self, dato: float) -> None:
        print(f'Temperatura: {dato} F')

sensor = Sensor()
dato = sensor.leer()
proc = Procesador()
resultado = proc.procesar(dato)
vis = Visualizador()
vis.mostrar(resultado)

Temperatura: 77.0 F


## Con patrón Facade (forma correcta)
El cliente interactúa con una sola clase que coordina todo el proceso.

In [2]:
class EstacionMeteorologica:
    def __init__(self) -> None:
        self.sensor = Sensor()
        self.proc = Procesador()
        self.vis = Visualizador()
    def mostrar_temperatura(self) -> None:
        dato = self.sensor.leer()
        resultado = self.proc.procesar(dato)
        self.vis.mostrar(resultado)

estacion = EstacionMeteorologica()
estacion.mostrar_temperatura()

Temperatura: 77.0 F


## UML del patrón Facade
```plantuml
@startuml
class EstacionMeteorologica {
    + mostrar_temperatura()
}
class Sensor {
    + leer()
}
class Procesador {
    + procesar(dato)
}
class Visualizador {
    + mostrar(dato)
}
EstacionMeteorologica --> Sensor
EstacionMeteorologica --> Procesador
EstacionMeteorologica --> Visualizador
@enduml
```

## Otro ejemplo de la vida real: Checkout de una tienda online
**Contexto:** al finalizar una compra, el backend de e-commerce debe: verificar stock en el inventario, cobrar el pago, generar la factura y programar el envío — cuatro subsistemas distintos que deben coordinarse en orden y detenerse si alguno falla (sin stock, o pago rechazado). El frontend/checkout no debería tener que conocer ni orquestar cada uno de esos subsistemas.

### Sin patrón (forma errónea)
El cliente coordina manualmente los 4 subsistemas y su orden de ejecución.

In [3]:
class Inventario:
    def verificar_stock(self, producto: str) -> bool:
        print(f'Verificando stock de {producto}...')
        return True

class PasarelaPago:
    def cobrar(self, monto: float) -> bool:
        print(f'Cobrando ${monto}...')
        return True

class Facturacion:
    def generar_factura(self, producto: str, monto: float) -> None:
        print(f'Generando factura de {producto} por ${monto}')

class ServicioEnvios:
    def programar_envio(self, producto: str) -> None:
        print(f'Programando envío de {producto}')

# El cliente debe conocer los 4 subsistemas y coordinarlos en el orden correcto
inventario = Inventario()
pago = PasarelaPago()
factura = Facturacion()
envios = ServicioEnvios()

if inventario.verificar_stock('Laptop'):
    if pago.cobrar(2500000):
        factura.generar_factura('Laptop', 2500000)
        envios.programar_envio('Laptop')

Verificando stock de Laptop...
Cobrando $2500000...
Generando factura de Laptop por $2500000
Programando envío de Laptop


### Con patrón (forma correcta)
`CheckoutFacade` expone un único método `procesar_compra()` que orquesta los 4 subsistemas y aplica las validaciones. El cliente ya no necesita saber que existen.

In [4]:
class CheckoutFacade:
    def __init__(self) -> None:
        self.inventario = Inventario()
        self.pago = PasarelaPago()
        self.factura = Facturacion()
        self.envios = ServicioEnvios()

    def procesar_compra(self, producto: str, monto: float) -> None:
        if not self.inventario.verificar_stock(producto):
            print('Sin stock disponible')
            return
        if not self.pago.cobrar(monto):
            print('Pago rechazado')
            return
        self.factura.generar_factura(producto, monto)
        self.envios.programar_envio(producto)


checkout = CheckoutFacade()
checkout.procesar_compra('Laptop', 2500000)

Verificando stock de Laptop...
Cobrando $2500000...
Generando factura de Laptop por $2500000
Programando envío de Laptop


### UML del ejemplo de checkout
```plantuml
@startuml
class CheckoutFacade {
    + procesar_compra(producto, monto)
}
class Inventario {
    + verificar_stock(producto)
}
class PasarelaPago {
    + cobrar(monto)
}
class Facturacion {
    + generar_factura(producto, monto)
}
class ServicioEnvios {
    + programar_envio(producto)
}
CheckoutFacade --> Inventario
CheckoutFacade --> PasarelaPago
CheckoutFacade --> Facturacion
CheckoutFacade --> ServicioEnvios
@enduml
```

### ¿Dónde más se usa Facade?
- **Checkout de e-commerce:** exactamente este ejemplo — orquestar inventario, pago, facturación y envío detrás de un solo endpoint.
- **SDKs de servicios en la nube:** el cliente de alto nivel de AWS/GCP oculta la complejidad de autenticación, reintentos y llamadas HTTP de bajo nivel detrás de métodos simples.
- **ORMs:** `session.commit()` en SQLAlchemy oculta la coordinación de transacciones, flush y sincronización con la base de datos.
- **Frameworks de testing:** un helper `crear_usuario_de_prueba()` que internamente coordina la creación en base de datos, el hash de contraseña y el envío de un email de bienvenida simulado.
- **Sistemas de reservas (vuelos, hoteles):** un método `reservar()` que coordina disponibilidad, cobro, confirmación y notificación, ocultando la complejidad de cada aerolínea/proveedor.

**Ejercicio de reflexión:** si `PasarelaPago.cobrar()` empezara a lanzar excepciones en vez de devolver `False`, ¿qué tendrías que cambiar en `CheckoutFacade` para mantener la misma interfaz simple hacia el cliente?

## Actividad
Crea tu propio Facade para simplificar el acceso a un sistema de reservas de vuelos o de compras online.

---
## Explicación de conceptos clave
- **Simplicidad:** El Facade oculta la complejidad interna y expone una interfaz sencilla.
- **Desacoplamiento:** El cliente no necesita conocer los detalles internos del sistema.
- **Aplicación en la vida real:** Útil en bibliotecas, APIs y sistemas complejos.

## Conclusión
El patrón Facade es ideal para simplificar el acceso a sistemas complejos y mejorar la experiencia del usuario. Es común en bibliotecas, APIs y aplicaciones con múltiples subsistemas.